# Tema 3 – Rețele Bayesiene
## Exercițiul 2 – Boala pulmonară

### Descrierea problemei

Un medic dorește să determine probabilitatea ca un pacient să aibă o boală pulmonară (`B`), pe baza simptomelor observate. Variabilele sunt binare:

- `B`: prezența bolii pulmonare;
- `T`: prezența tusei;
- `D`: dificultate de respirație;
- `X`: radiografie anormală.

Probabilitățile date în enunț sunt:

- \(P(B=1)=0.1\), \(P(B=0)=0.9\);
- \(P(T=1\mid B=1)=0.8\), \(P(T=1\mid B=0)=0.3\);
- \(P(X=1\mid B=1)=0.9\), \(P(X=1\mid B=0)=0.1\);
- \(P(D=1\mid B=1,T=1)=0.9\);
- \(P(D=1\mid B=1,T=0)=0.4\);
- \(P(D=1\mid B=0,T=1)=0.5\);
- \(P(D=1\mid B=0,T=0)=0.1\).

Vom construi o rețea Bayesiană, vom calcula probabilitățile cerute și vom verifica rezultatele folosind regula lui Bayes.

## a) Definirea modelului probabilist cu `pgmpy`

Structura rețelei este:

- `B → T`: tusea depinde de boală;
- `B → X`: rezultatul radiografiei depinde de boală;
- `B → D` și `T → D`: dificultatea de respirație depinde de boală și de tuse.

Graficul exprimă independențele condiționale presupuse în enunț.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Structura rețelei
model = DiscreteBayesianNetwork([
    ("B", "T"),
    ("B", "X"),
    ("B", "D"),
    ("T", "D")
])

# P(B): probabilitatea apriori a bolii
cpd_b = TabularCPD(
    variable="B",
    variable_card=2,
    values=[[0.9], [0.1]
])

# P(T | B)
# Rândurile corespund lui T=0 și T=1.
# Coloanele corespund lui B=0 și B=1.
cpd_t = TabularCPD(
    variable="T",
    variable_card=2,
    values=[
        [0.7, 0.2],  # P(T=0 | B)
        [0.3, 0.8]   # P(T=1 | B)
    ],
    evidence=["B"],
    evidence_card=[2]
)

# P(X | B)
# Coloanele corespund lui B=0 și B=1.
cpd_x = TabularCPD(
    variable="X",
    variable_card=2,
    values=[
        [0.9, 0.1],  # P(X=0 | B)
        [0.1, 0.9]   # P(X=1 | B)
    ],
    evidence=["B"],
    evidence_card=[2]
)

# P(D | B, T)
# Ordinea coloanelor pentru evidence=["B", "T"]:
# (B=0,T=0), (B=0,T=1), (B=1,T=0), (B=1,T=1)
cpd_d = TabularCPD(
    variable="D",
    variable_card=2,
    values=[
        [0.9, 0.5, 0.6, 0.1],  # P(D=0 | B,T)
        [0.1, 0.5, 0.4, 0.9]   # P(D=1 | B,T)
    ],
    evidence=["B", "T"],
    evidence_card=[2, 2]
)

# Adăugăm distribuțiile în rețea
model.add_cpds(cpd_b, cpd_t, cpd_x, cpd_d)

# Verificăm dacă modelul este valid
print("Model valid:", model.check_model())

### Explicația codului

1. `DiscreteBayesianNetwork` definește graful și relațiile dintre variabile.
2. `TabularCPD` definește distribuțiile de probabilitate condiționată.
3. `cpd_b` conține probabilitatea apriori a bolii.
4. `cpd_t` descrie probabilitatea de a avea tuse în funcție de prezența bolii.
5. `cpd_x` descrie probabilitatea unei radiografii anormale în funcție de boală.
6. `cpd_d` descrie dificultatea de respirație în funcție de boală și tuse.
7. `model.check_model()` verifică dacă distribuțiile sunt compatibile cu structura rețelei.

În fiecare tabel, rândul cu valoarea 0 reprezintă probabilitatea ca variabila să fie falsă, iar rândul cu valoarea 1 reprezintă probabilitatea ca variabila să fie adevărată. Pentru fiecare combinație a părinților, probabilitățile dintr-o coloană trebuie să însumeze 1.

## b) Probabilitatea bolii când pacientul are tuse și dificultăți de respirație

Trebuie să calculăm:

\[
P(B=1\mid T=1,D=1).
\]

Radiografia `X` nu este observată în această cerință, deci nu o folosim ca dovadă.

In [ ]:
infer = VariableElimination(model)

posterior_b = infer.query(
    variables=["B"],
    evidence={"T": 1, "D": 1}
)

print(posterior_b)

### Interpretarea rezultatului

Valoarea pentru `B=1` din distribuția afișată reprezintă probabilitatea ca pacientul să aibă boala, știind că are tuse și dificultăți de respirație.

In [ ]:
prob_b = float(posterior_b.values[1])
print(f"P(B=1 | T=1, D=1) = {prob_b:.6f}")
print(f"Procent: {prob_b * 100:.2f}%")

## c) Probabilitatea unei radiografii anormale dacă pacientul nu are boala

Cerința este:

\[
P(X=1\mid B=0).
\]

Conform datelor din enunț, această probabilitate este direct \(0.1\). O verificăm și cu rețeaua Bayesiană.

In [ ]:
posterior_x = infer.query(
    variables=["X"],
    evidence={"B": 0}
)

print(posterior_x)
print(f"P(X=1 | B=0) = {float(posterior_x.values[1]):.6f}")

## d) Verificarea manuală folosind regula lui Bayes

### Prima probabilitate: \(P(B=1\mid T=1,D=1)\)

Aplicăm formula lui Bayes:

\[
P(B=1\mid T=1,D=1)=
\frac{P(B=1,T=1,D=1)}
{P(T=1,D=1)}.
\]

Din structura rețelei, pentru valorile observate ale lui `T` și `D`, probabilitatea comună se calculează astfel:

\[
P(B,T=1,D=1)=P(B)P(T=1\mid B)P(D=1\mid B,T=1).
\]

Pentru \(B=1\):

\[
P(B=1,T=1,D=1)=0.1\cdot0.8\cdot0.9=0.072.
\]

Pentru \(B=0\):

\[
P(B=0,T=1,D=1)=0.9\cdot0.3\cdot0.5=0.135.
\]

Prin urmare:

\[
P(T=1,D=1)=0.072+0.135=0.207.
\]

Rezultă:

\[
P(B=1\mid T=1,D=1)=\frac{0.072}{0.207}\approx0.347826.
\]

Deci probabilitatea ca pacientul să aibă boala, având tuse și dificultăți de respirație, este de aproximativ **34.78%**.

### A doua probabilitate: \(P(X=1\mid B=0)\)

Această probabilitate este dată direct în enunț:

\[
P(X=1\mid B=0)=0.1.
\]

Folosind independența condițională din rețea, `X` depinde doar de `B`. Prin urmare, după ce știm că \(B=0\), informațiile despre celelalte simptome nu schimbă această probabilitate.

Rezultatul este **0.1, adică 10%**.

## Concluzie

Am definit o rețea Bayesiană cu patru variabile binare: boala pulmonară (`B`), tusea (`T`), dificultatea de respirație (`D`) și radiografia anormală (`X`). Rețeaua respectă dependențele din enunț.

- Pentru un pacient cu tuse și dificultăți de respirație, probabilitatea bolii este aproximativ **34.78%**.
- Dacă știm că pacientul nu are boala, probabilitatea ca radiografia să fie anormală este **10%**.

Rezultatele au fost obținute cu `pgmpy` și verificate prin calcule folosind regula lui Bayes.